# E7 — Calendar Spread Daily Stats W2/W3/W4

In [1]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [4]:
daily_rows = []
for wk in ['W2','W3','W4']:
    wm = WINDOWS_META[wk]
    for day in wm['days']:
        mids = {}
        for sym in [wm['front'], wm['back']]:
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            if not fpath: continue
            df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
            df['ts_event'] = pd.to_datetime(df['ts_event'])
            rth = ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) >= wm['rth_start_min']) & \
                  ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) <= wm['rth_end_min'])
            df = df[rth].set_index('ts_event')
            mids[sym] = ((df['bid_px_00']+df['ask_px_00'])/2).resample('1min').last().dropna()
            del df; gc.collect()
        if len(mids) < 2: continue
        cal = mids[wm['back']].subtract(mids[wm['front']], fill_value=np.nan).dropna()
        if len(cal) < 2: continue
        daily_rows.append(dict(window=wk, day=day,
                               cal_open=cal.iloc[0], cal_close=cal.iloc[-1],
                               cal_drift=cal.iloc[-1]-cal.iloc[0],
                               cal_ba_mean=np.nan))
daily_df = pd.DataFrame(daily_rows)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for row, wk in enumerate(['W2','W3','W4']):
    sub = daily_df[daily_df['window']==wk].reset_index(drop=True)
    x = range(len(sub))
    wm = WINDOWS_META[wk]
    # Get labels only for days with data
    day_indices = [wm['days'].index(d) for d in sub['day']]
    labels = [wm['day_labels'][i] for i in day_indices]
    # levels
    axes[row][0].plot(x, sub['cal_open'],  marker='o', label='Open',  color='steelblue')
    axes[row][0].plot(x, sub['cal_close'], marker='s', label='Close', color='darkorange')
    axes[row][0].set_title(f'{wk} Spread Levels')
    axes[row][0].legend(fontsize=8)
    # drift
    colors = ['#2ecc71' if d >= 0 else '#e74c3c' for d in sub['cal_drift']]
    axes[row][1].bar(x, sub['cal_drift'], color=colors, alpha=0.85)
    axes[row][1].axhline(0, color='black', lw=0.5)
    axes[row][1].set_title(f'{wk} Daily Drift')
    # xticks
    for col in range(3):
        axes[row][col].set_xticks(list(x))
        axes[row][col].set_xticklabels(labels, fontsize=7)
    axes[row][2].set_visible(False)  # cal_ba placeholder
fig.suptitle('Calendar Spread Daily Stats — W2/W3/W4', fontsize=13)
save_fig(fig, 'E7_cal_spread_daily_w2w3w4.png')

  Saved --> figures/E7_cal_spread_daily_w2w3w4.png
